In [5]:
import pandas as pd
vehicles = pd.read_excel("./Datasets/Vehicles.xlsx")
telemetry= pd.read_excel("./Datasets/Telemetry.xlsx")
trips = pd.read_excel("./Datasets/Trips.xlsx")

In [6]:
telemetry["Timestamp"] = pd.to_datetime(telemetry["Timestamp"])
vehicles["Last_Service_Date"] = pd.to_datetime(
    vehicles["Last_Service_Date"]
)


In [8]:
import numpy as np

telemetry["Acceleration_Magnitude"] = np.sqrt(
    telemetry["Accel_X_g"]**2 +
    telemetry["Accel_Y_g"]**2 +
    telemetry["Accel_Z_g"]**2
)

telemetry["Gyro_Magnitude"] = np.sqrt(
    telemetry["Gyro_X_dps"]**2 +
    telemetry["Gyro_Y_dps"]**2 +
    telemetry["Gyro_Z_dps"]**2
)

In [9]:
accel_threshold = (
    telemetry["Acceleration_Magnitude"].mean()
    + 3 * telemetry["Acceleration_Magnitude"].std()
)

gyro_threshold = (
    telemetry["Gyro_Magnitude"].mean()
    + 3 * telemetry["Gyro_Magnitude"].std()
)

telemetry["Accel_Abnormal"] = (
    telemetry["Acceleration_Magnitude"] > accel_threshold
)

telemetry["Gyro_Abnormal"] = (
    telemetry["Gyro_Magnitude"] > gyro_threshold
)

telemetry["Abnormal"] = (
    telemetry["Accel_Abnormal"] |
    telemetry["Gyro_Abnormal"]
)

In [10]:
vehicle_sensor = telemetry.groupby("Vehicle_ID").agg(

    Total_Readings=("Vehicle_ID", "size"),

    Abnormal_Readings=("Abnormal", "sum"),

    Avg_Acceleration=("Acceleration_Magnitude", "mean"),

    Max_Acceleration=("Acceleration_Magnitude", "max"),

    Avg_Gyro=("Gyro_Magnitude", "mean"),

    Max_Gyro=("Gyro_Magnitude", "max")

).reset_index()

In [11]:
vehicle_sensor["Abnormal_Percentage"] = (
    vehicle_sensor["Abnormal_Readings"]
    / vehicle_sensor["Total_Readings"]
    * 100
)

In [12]:
result = vehicle_sensor.merge(
    vehicles,
    on="Vehicle_ID",
    how="left"
)

result["Vehicle_Age"] = (
    2026 - result["Manufacture_Year"]
)

result["Days_Since_Service"] = (
    pd.Timestamp("2026-08-12")
    - result["Last_Service_Date"]
).dt.days


In [13]:
def maintenance_risk(row):

    score = 0

    # Sensor abnormalities
    if row["Abnormal_Percentage"] > 5:
        score += 2
    elif row["Abnormal_Percentage"] > 2:
        score += 1

    # Vehicle age
    if row["Vehicle_Age"] >= 7:
        score += 2
    elif row["Vehicle_Age"] >= 4:
        score += 1

    # Mileage
    if row["Odometer_KM_Start_of_Week"] >= 100000:
        score += 2
    elif row["Odometer_KM_Start_of_Week"] >= 50000:
        score += 1

    # Service
    if row["Days_Since_Service"] >= 180:
        score += 2
    elif row["Days_Since_Service"] >= 90:
        score += 1

    if score >= 5:
        return "High"
    elif score >= 2:
        return "Medium"
    else:
        return "Low"


result["Maintenance_Risk"] = result.apply(
    maintenance_risk,
    axis=1
)


In [14]:
result = result.sort_values(
    "Abnormal_Percentage",
    ascending=False
)

In [15]:
result.to_excel(
    "Vehicle_Maintenance_Analysis.xlsx",
    index=False
)

print(result.head())

   Vehicle_ID  Total_Readings  Abnormal_Readings  Avg_Acceleration  \
1         V02             443                 42          1.047501   
22        V23             555                 48          1.037333   
18        V19             374                 32          1.042361   
0         V01             427                 34          1.024864   
13        V14             446                 32          1.035942   

    Max_Acceleration  Avg_Gyro   Max_Gyro  Abnormal_Percentage  \
1           1.813246  4.920702  52.919581             9.480813   
22          1.776017  5.574218  57.619533             8.648649   
18          1.879035  5.284249  52.626751             8.556150   
0           1.639733  5.442126  54.095156             7.962529   
13          1.967638  5.070825  50.062635             7.174888   

                        Vehicle_Type    Make   Model  Manufacture_Year  \
1   Two-Wheeler (Scooter/Motorcycle)     TVS  Raider              2021   
22  Two-Wheeler (Scooter/Motorcycl

In [16]:
df = pd.read_excel("Vehicle_Maintenance_Analysis.xlsx")
df.head()

,Vehicle_ID,Total_Readings,Abnormal_Readings,Avg_Acceleration,Max_Acceleration,Avg_Gyro,Max_Gyro,Abnormal_Percentage,Vehicle_Type,Make,Model,Manufacture_Year,Registration_Date,Odometer_KM_Start_of_Week,Last_Service_Date,Vehicle_Age,Days_Since_Service,Maintenance_Risk
0,V02,443,42,1.047501,1.813246,4.920702,52.919581,9.480813,Two-Wheeler (Scooter/Motorcycle),TVS,Raider,2021,2021-03-26,46601,2026-05-06,5,98,Medium
1,V23,555,48,1.037333,1.776017,5.574218,57.619533,8.648649,Two-Wheeler (Scooter/Motorcycle),Suzuki,Access,2021,2021-01-27,17821,2026-06-16,5,57,Medium
2,V19,374,32,1.042361,1.879035,5.284249,52.626751,8.556150,Two-Wheeler (Scooter/Motorcycle),TVS,Ntorq,2021,2021-12-09,17247,2026-06-19,5,54,Medium
3,V01,427,34,1.024864,1.639733,5.442126,54.095156,7.962529,Two-Wheeler (Scooter/Motorcycle),Yamaha,Ray ZR,2021,2021-04-23,22820,2026-06-08,5,65,Medium
4,V14,446,32,1.035942,1.967638,5.070825,50.062635,7.174888,Two-Wheeler (Scooter/Motorcycle),TVS,Raider,2019,2019-03-07,5270,2026-07-26,7,17,Medium
